# 05 · Live training plots (from the metrics log)
Reads `runs/<name>/metrics.jsonl` and plots train/val loss, learning rate, and gradient norm. **Re-run any time — even mid-training** — to refresh the curves (the trainer flushes every step).

In [ ]:
# --- Bootstrap: make the package importable without installing, and stay OFFLINE.
import os, sys
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
print("repo root:", ROOT)


In [ ]:
import json
import matplotlib.pyplot as plt
from gemma_ft_json.config import load_config

cfg = load_config(ROOT / 'configs' / 'default.yaml')
run_dir = ROOT / 'runs' / cfg.project.name
metrics_path = run_dir / cfg.logging.metrics_filename
assert metrics_path.is_file(), f'No metrics yet at {metrics_path} — run notebook 04 first.'
rows = [json.loads(l) for l in open(metrics_path).read().splitlines() if l.strip()]
train = [r for r in rows if r.get('event') == 'train_step']
val   = [r for r in rows if r.get('event') == 'val']
print(f'{len(train)} train points, {len(val)} val points')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
# Loss (train + val)
axes[0].plot([r['step'] for r in train], [r['loss'] for r in train], label='train', color='#4C72B0')
if val:
    axes[0].plot([r['step'] for r in val], [r['val_loss'] for r in val], 'o-', label='val', color='#C44E52')
axes[0].set_xlabel('optimizer step'); axes[0].set_ylabel('loss'); axes[0].set_title('Loss'); axes[0].legend()
# Learning rate
axes[1].plot([r['step'] for r in train], [r['lr'] for r in train], color='#55A868')
axes[1].set_xlabel('step'); axes[1].set_ylabel('lr'); axes[1].set_title('LR schedule (warmup→cosine)')
# Gradient norm (if present)
gn = [(r['step'], r.get('grad_norm')) for r in train if r.get('grad_norm') is not None]
if gn:
    axes[2].plot([s for s, _ in gn], [g for _, g in gn], color='#8172B3')
axes[2].set_xlabel('step'); axes[2].set_ylabel('grad norm'); axes[2].set_title('Gradient norm')
plt.tight_layout(); plt.show()

Tip: for a true *live* dashboard, run this notebook in one window and notebook 04 in another; re-execute the plot cell periodically (or wrap it in a loop with `time.sleep`).